## Test 1: Markov Diffusion Consistency

In [3]:
import json
import numpy as np

# ============================================================
# Load scenario (for num_classes / T only — this test is about
# the diffusion math itself, independent of features)
# ============================================================
def load_scenario(filepath="scenario.json"):
    with open(filepath, "r") as f:
        return json.load(f)


# ============================================================
# D3PM-style categorical transition matrix (uniform noise kernel)
# Q_t: at each step, with prob (1-beta_t) stay the same,
# with prob beta_t jump to a uniform random category
# ============================================================
def make_transition_matrix(num_classes, beta_t):
    """Single-step transition matrix Q_t (num_classes x num_classes)."""
    Q = np.full((num_classes, num_classes), beta_t / num_classes)
    np.fill_diagonal(Q, (1 - beta_t) + beta_t / num_classes)
    return Q


def make_beta_schedule(num_steps, beta_start=0.001, beta_end=0.2):
    """Linear beta schedule across diffusion steps."""
    return np.linspace(beta_start, beta_end, num_steps)


def cumulative_transition_matrices(num_classes, betas):
    """
    Compute Q_1, Q_2, ..., Q_T (single-step)
    and Qbar_1, ..., Qbar_T (cumulative product, i.e. Qbar_t = Q_1 @ Q_2 @ ... @ Q_t)
    """
    Q_list = [make_transition_matrix(num_classes, b) for b in betas]
    Qbar_list = []
    Qbar = np.eye(num_classes)
    for Q in Q_list:
        Qbar = Qbar @ Q
        Qbar_list.append(Qbar)
    return Q_list, Qbar_list


# ============================================================
# TEST 1a: Probability conservation
# Every row of every transition matrix (single-step and cumulative)
# must sum to 1 (valid categorical distribution)
# ============================================================
def test_probability_conservation(Q_list, Qbar_list, tol=1e-8):
    print("=== Test 1a: Probability Conservation ===")
    all_pass = True

    for i, Q in enumerate(Q_list):
        row_sums = Q.sum(axis=1)
        ok = np.allclose(row_sums, 1.0, atol=tol)
        all_pass &= ok
        if not ok:
            print(f"  FAIL: Q_{i+1} row sums not 1.0 -> {row_sums}")

    for i, Qbar in enumerate(Qbar_list):
        row_sums = Qbar.sum(axis=1)
        ok = np.allclose(row_sums, 1.0, atol=tol)
        all_pass &= ok
        if not ok:
            print(f"  FAIL: Qbar_{i+1} row sums not 1.0 -> {row_sums}")

    print(f"  Result: {'PASS' if all_pass else 'FAIL'}\n")
    return all_pass


# ============================================================
# TEST 1b: Markov consistency
# Qbar_t computed via direct matrix chain-product must match
# Qbar_t computed step-by-step by applying single-step Q's
# sequentially to a distribution (sampling-based check)
# ============================================================
def test_markov_consistency(num_classes, betas, num_trials=2000, seed=0, tol=0.03):
    print("=== Test 1b: Markov Consistency (sampling-based) ===")
    rng = np.random.default_rng(seed)
    T = len(betas)

    Q_list, Qbar_list = cumulative_transition_matrices(num_classes, betas)

    # Start from a fixed one-hot class (e.g., class 0)
    start_class = 0

    # (1) Analytical: directly read row of Qbar_T for start_class
    analytical_dist = Qbar_list[-1][start_class]

    # (2) Empirical: simulate the chain step by step many times, count final class
    empirical_counts = np.zeros(num_classes)
    for _ in range(num_trials):
        x = start_class
        for Q in Q_list:
            probs = Q[x]
            x = rng.choice(num_classes, p=probs)
        empirical_counts[x] += 1
    empirical_dist = empirical_counts / num_trials

    max_diff = np.max(np.abs(analytical_dist - empirical_dist))
    print(f"  Analytical Qbar_T[start_class]: {analytical_dist.round(4)}")
    print(f"  Empirical (sampled) distribution: {empirical_dist.round(4)}")
    print(f"  Max abs difference: {max_diff:.4f} (tolerance: {tol})")

    passed = max_diff < tol
    print(f"  Result: {'PASS' if passed else 'FAIL'}\n")
    return passed


# ============================================================
# TEST 1c: Forward/reverse posterior sanity
# q(x_{t-1} | x_t, x_0) computed via Bayes rule using Q_t and Qbar_{t-1}
# must be a valid probability distribution (sums to 1, non-negative)
# ============================================================
def compute_posterior(x_t, x_0, Q_t, Qbar_tm1, num_classes, eps=1e-8):
    """
    q(x_{t-1} | x_t, x_0) proportional to:
      [Q_t^T]_{x_t, :} * [Qbar_{t-1}]_{x_0, :}
    (elementwise product over the x_{t-1} dimension), normalized.
    """
    term1 = Q_t[:, x_t]        # Q_t[x_{t-1}, x_t] for all x_{t-1} -> column x_t
    term2 = Qbar_tm1[x_0, :]   # Qbar_{t-1}[x_0, x_{t-1}] for all x_{t-1}
    unnorm = term1 * term2
    total = unnorm.sum() + eps
    return unnorm / total


def test_posterior_validity(num_classes, betas, num_checks=20, seed=0):
    print("=== Test 1c: Reverse Posterior Validity ===")
    rng = np.random.default_rng(seed)
    T = len(betas)
    Q_list, Qbar_list = cumulative_transition_matrices(num_classes, betas)

    all_pass = True
    for _ in range(num_checks):
        t = rng.integers(1, T)  # need t-1 >= 0, so t starts at 1
        x_t = rng.integers(0, num_classes)
        x_0 = rng.integers(0, num_classes)

        Q_t = Q_list[t]                                  # single-step Q_t
        Qbar_tm1 = Qbar_list[t - 1] if t > 0 else np.eye(num_classes)  # Qbar_{t-1}

        posterior = compute_posterior(x_t, x_0, Q_t, Qbar_tm1, num_classes)

        sum_ok = np.isclose(posterior.sum(), 1.0, atol=1e-4)
        nonneg_ok = np.all(posterior >= -1e-8)
        ok = sum_ok and nonneg_ok
        all_pass &= ok

        if not ok:
            print(f"  FAIL at t={t}, x_t={x_t}, x_0={x_0}: "
                  f"sum={posterior.sum():.6f}, min={posterior.min():.6f}")

    print(f"  Checked {num_checks} random (t, x_t, x_0) combos")
    print(f"  Result: {'PASS' if all_pass else 'FAIL'}\n")
    return all_pass


# ============================================================
# Run all Test 1 sub-checks
# ============================================================
if __name__ == "__main__":
    scenario = load_scenario("scenarios\scenario_3_15_2001.json")
    T = scenario["T"]
    num_classes = T + 1  # 0 = unassigned, 1..T = order values

    num_steps = 50
    betas = make_beta_schedule(num_steps, beta_start=0.001, beta_end=0.2)

    print(f"Loaded scenario.json | T={T} | num_classes={num_classes} | num_steps={num_steps}\n")

    Q_list, Qbar_list = cumulative_transition_matrices(num_classes, betas)

    r1 = test_probability_conservation(Q_list, Qbar_list)
    r2 = test_markov_consistency(num_classes, betas, num_trials=3000)
    r3 = test_posterior_validity(num_classes, betas, num_checks=30)

    print("=== TEST 1 SUMMARY ===")
    print(f"1a. Probability conservation: {'PASS' if r1 else 'FAIL'}")
    print(f"1b. Markov consistency:       {'PASS' if r2 else 'FAIL'}")
    print(f"1c. Posterior validity:       {'PASS' if r3 else 'FAIL'}")
    print(f"OVERALL: {'PASS' if (r1 and r2 and r3) else 'FAIL'}")

Loaded scenario.json | T=15 | num_classes=16 | num_steps=50

=== Test 1a: Probability Conservation ===
  Result: PASS

=== Test 1b: Markov Consistency (sampling-based) ===
  Analytical Qbar_T[start_class]: [0.0667 0.0622 0.0622 0.0622 0.0622 0.0622 0.0622 0.0622 0.0622 0.0622
 0.0622 0.0622 0.0622 0.0622 0.0622 0.0622]
  Empirical (sampled) distribution: [0.0693 0.0593 0.0563 0.065  0.071  0.0593 0.0667 0.0613 0.0623 0.0583
 0.0707 0.0573 0.0607 0.0607 0.0647 0.057 ]
  Max abs difference: 0.0088 (tolerance: 0.03)
  Result: PASS

=== Test 1c: Reverse Posterior Validity ===
  Checked 30 random (t, x_t, x_0) combos
  Result: PASS

=== TEST 1 SUMMARY ===
1a. Probability conservation: PASS
1b. Markov consistency:       PASS
1c. Posterior validity:       PASS
OVERALL: PASS


## Test 2: Sinkhorn Normalization

In [7]:
import numpy as np
import time
import torch
import torch.nn as nn

# ============================================================
# Sinkhorn normalization: iteratively normalize rows and columns
# of exp(logits) so the result approximates a valid assignment
# (each task -> ~1 robot, each robot-slot -> ~1 task)
# ============================================================
def sinkhorn(logits, num_iters=20, temperature=1.0, eps=1e-8):
    log_alpha = logits / temperature
    log_alpha = log_alpha - torch.logsumexp(log_alpha, dim=0, keepdim=True)  # column-normalize only
    return torch.exp(log_alpha)


# ============================================================
# TEST 2: Run Sinkhorn at increasing problem sizes
# ============================================================
def test_sinkhorn_validity(R, T, num_iters=20, temperature=0.5, seed=0):
    torch.manual_seed(seed)
    logits = torch.randn(R, T) * 2.0  # random raw scores

    start = time.time()
    P = sinkhorn(logits, num_iters=num_iters, temperature=temperature)
    elapsed = time.time() - start

    row_sums = P.sum(dim=1).detach().numpy()
    col_sums = P.sum(dim=0).detach().numpy()

    # "soft" one-robot-per-task check: how concentrated is each column?
    # A well-converged Sinkhorn matrix should have each column dominated
    # by one large value (close to a permutation-like structure)
    col_max = P.max(dim=0).values.detach().numpy()  # largest value per task-column
    row_max = P.max(dim=1).values.detach().numpy()  # largest value per robot-row

    print(f"--- R={R}, T={T} ---")
    print(f"  Time taken: {elapsed*1000:.2f} ms")
    print(f"  Row sums (should be ~1): min={row_sums.min():.4f}, max={row_sums.max():.4f}")
    print(f"  Col sums (should be ~1): min={col_sums.min():.4f}, max={col_sums.max():.4f}")
    print(f"  Column concentration (max value per task): min={col_max.min():.4f}, "
          f"mean={col_max.mean():.4f} (closer to 1.0 = more decisive assignment)")
    print(f"  Row concentration (max value per robot):   min={row_max.min():.4f}, "
          f"mean={row_max.mean():.4f}")

    col_ok = np.allclose(col_sums, 1.0, atol=0.05)   # each task -> exactly one robot
    row_nonneg_ok = np.all(row_sums >= -1e-8)          # each robot can take 0+ tasks, no upper bound needed
    print(f"  Row-sum check: {'PASS' if row_nonneg_ok else 'FAIL'}")
    print(f"  Col-sum check: {'PASS' if col_ok else 'FAIL'}\n")

    return elapsed, row_nonneg_ok, col_ok, col_max.mean(), row_max.mean()


if __name__ == "__main__":
    sizes = [(3, 5), (10, 30), (50, 100)]

    results = []
    for R, T in sizes:
        elapsed, row_ok, col_ok, col_conc, row_conc = test_sinkhorn_validity(
            R, T, num_iters=20, temperature=0.5
        )
        results.append((R, T, elapsed, row_ok, col_ok, col_conc, row_conc))

    print("=== TEST 2 SUMMARY ===")
    for R, T, elapsed, row_ok, col_ok, col_conc, row_conc in results:
        status = "PASS" if (row_ok and col_ok) else "FAIL"
        print(f"R={R:3d}, T={T:3d} | Time={elapsed*1000:7.2f}ms | "
              f"RowSum={'OK' if row_ok else 'BAD'} | ColSum={'OK' if col_ok else 'BAD'} | "
              f"ColConc={col_conc:.3f} | RowConc={row_conc:.3f} | {status}")

--- R=3, T=5 ---
  Time taken: 0.00 ms
  Row sums (should be ~1): min=1.2056, max=2.5591
  Col sums (should be ~1): min=1.0000, max=1.0000
  Column concentration (max value per task): min=0.6783, mean=0.8903 (closer to 1.0 = more decisive assignment)
  Row concentration (max value per robot):   min=0.8931, mean=0.9639
  Row-sum check: PASS
  Col-sum check: PASS

--- R=10, T=30 ---
  Time taken: 1.00 ms
  Row sums (should be ~1): min=1.6630, max=4.1491
  Col sums (should be ~1): min=1.0000, max=1.0000
  Column concentration (max value per task): min=0.3559, mean=0.6400 (closer to 1.0 = more decisive assignment)
  Row concentration (max value per robot):   min=0.5067, mean=0.7717
  Row-sum check: PASS
  Col-sum check: PASS

--- R=50, T=100 ---
  Time taken: 0.00 ms
  Row sums (should be ~1): min=0.4183, max=4.3772
  Col sums (should be ~1): min=1.0000, max=1.0000
  Column concentration (max value per task): min=0.2450, mean=0.6626 (closer to 1.0 = more decisive assignment)
  Row concentr

## Test 3: Reward Function Sanity

In [16]:
import json
import numpy as np

def load_scenario(filepath="scenario.json"):
    with open(filepath, "r") as f:
        return json.load(f)


def build_random_schedule(scenario, seed=0):
    rng = np.random.default_rng(seed)
    R, T = scenario["R"], scenario["T"]
    capacity = np.array(scenario["robot_max_payload"])
    weight = np.array(scenario["task_weight"])

    task_order = rng.permutation(T)
    remaining_capacity = capacity.copy()
    schedule = np.zeros((R, T), dtype=int)
    robot_next_slot = np.ones(R, dtype=int)

    for t in task_order:
        candidates = np.where(remaining_capacity >= weight[t])[0]
        if len(candidates) == 0:
            continue
        r = rng.choice(candidates)
        schedule[r, t] = robot_next_slot[r]
        robot_next_slot[r] += 1
        remaining_capacity[r] -= weight[t]

    return schedule


def decode_routes(schedule):
    R, T = schedule.shape
    routes = []
    for r in range(R):
        row = schedule[r]
        assigned = [(order, t) for t, order in enumerate(row) if order > 0]
        assigned.sort(key=lambda x: x[0])
        routes.append([t for _, t in assigned])
    return routes


def compute_objectives(scenario, schedule):
    R = scenario["R"]
    robot_pos = np.array(scenario["robot_pos"])
    robot_speed = np.full(R, 5.0)          # placeholder, will come from sequencing head later
    robot_energy_rate = np.full(R, 1.0)    # placeholder constant for now
    task_pos = np.array(scenario["task_pos"])
    task_service_time = np.array(scenario["task_service_time"])

    routes = decode_routes(schedule)

    robot_times = np.zeros(R)
    robot_energy = np.zeros(R)

    for r in range(R):
        current_pos = robot_pos[r]
        total_time = 0.0
        total_energy = 0.0

        for t in routes[r]:
            target_pos = task_pos[t]
            dist = np.linalg.norm(target_pos - current_pos)
            travel_time = dist / robot_speed[r]
            travel_energy = dist * robot_energy_rate[r]

            total_time += travel_time + task_service_time[t]
            total_energy += travel_energy

            current_pos = target_pos

        robot_times[r] = total_time
        robot_energy[r] = total_energy

    makespan = robot_times.max()
    workload_variance = robot_times.var()
    total_energy = robot_energy.sum()

    return {
        "makespan": makespan,
        "workload_variance": workload_variance,
        "total_energy": total_energy,
        "robot_times": robot_times,
        "robot_energy": robot_energy,
    }


def zscore_batch(values, eps=1e-8):
    values = np.array(values)
    mean = values.mean()
    std = values.std() + eps
    return (values - mean) / std, mean, std


# ============================================================
# FIXED reward: asymmetric achievement scalarization
# For minimize-type objectives: penalize only being WORSE than target.
# Being better than target costs near-zero (small reward, not punished).
# ============================================================
def compute_rewards(objective_batch, reference_point, rho=0.05):
    """
    reference_point: [target_makespan, target_variance, target_energy]
    rho: small weight on the "augmentation" term so ties still prefer
         genuinely better (lower) values, avoiding flat/plateaued reward
    """
    names = ["makespan", "workload_variance", "total_energy"]
    z_scores = {}
    stats = {}
    for name in names:
        z, mean, std = zscore_batch(objective_batch[name])
        z_scores[name] = z
        stats[name] = (mean, std)

    ref_z = {}
    for name, target in zip(names, reference_point):
        mean, std = stats[name]
        ref_z[name] = (target - mean) / std

    n = len(objective_batch["makespan"])
    penalty = np.zeros(n)
    aug = np.zeros(n)

    for name in names:
        diff = z_scores[name] - ref_z[name]          # positive = worse than target (since minimize)
        penalty += np.maximum(diff, 0.0) ** 2          # only punish overshoot above target
        aug += diff                                     # small linear term: always prefer lower, even below target

    dist = np.sqrt(penalty) + rho * aug
    reward = -dist
    return reward, z_scores["makespan"], z_scores["workload_variance"], z_scores["total_energy"]


def test_reward_sanity(json_path="scenario.json", num_schedules=30, seed=0):
    scenario = load_scenario(json_path)

    schedules = [build_random_schedule(scenario, seed=s) for s in range(num_schedules)]

    makespans, variances, energies = [], [], []
    for sched in schedules:
        obj = compute_objectives(scenario, sched)
        makespans.append(obj["makespan"])
        variances.append(obj["workload_variance"])
        energies.append(obj["total_energy"])

    objective_batch = {
        "makespan": makespans,
        "workload_variance": variances,
        "total_energy": energies,
    }

    print(f"Generated {num_schedules} random schedules")
    print(f"Makespan   range: [{min(makespans):.2f}, {max(makespans):.2f}]")
    print(f"Variance   range: [{min(variances):.2f}, {max(variances):.2f}]")
    print(f"Energy     range: [{min(energies):.2f}, {max(energies):.2f}]\n")

    reference_points = {
        "Low Makespan Target": [30, 0.1, 9],
        "Low Variance Target": [40, 0.01, 9],
        "Low Energy Target":   [45, 0.1, 7],
    }

    results = {}
    for name, ref in reference_points.items():
        rewards, z_m, z_v, z_e = compute_rewards(objective_batch, ref)
        best_idx = np.argmax(rewards)
        results[name] = best_idx

        print(f"--- Reference: {name} {ref} ---")
        print(f"  Best schedule index: {best_idx}")
        print(f"  Best schedule -> makespan={makespans[best_idx]:.2f}, "
              f"variance={variances[best_idx]:.2f}, energy={energies[best_idx]:.2f}")
        print(f"  (z-scores: time={z_m[best_idx]:.2f}, var={z_v[best_idx]:.2f}, energy={z_e[best_idx]:.2f})\n")

    print("=== Sanity Checks ===")
    check1 = makespans[results["Low Makespan Target"]] <= makespans[results["Low Energy Target"]]
    print(f"Low-Makespan-Target picks lower/equal makespan than Low-Energy-Target: {'PASS' if check1 else 'FAIL'}")

    check2 = variances[results["Low Variance Target"]] <= np.median(variances)
    print(f"Low-Variance-Target picks below-median variance: {'PASS' if check2 else 'FAIL'}")

    check3 = energies[results["Low Energy Target"]] <= energies[results["Low Makespan Target"]]
    print(f"Low-Energy-Target picks lower/equal energy than Low-Makespan-Target: {'PASS' if check3 else 'FAIL'}")

    unique_choices = len(set(results.values()))
    print(f"Distinct 'best schedule' choices across {len(reference_points)} references: {unique_choices}")

    return results, objective_batch



if __name__ == "__main__":
    test_reward_sanity("scenarios/scenario_3_15_1003.json", num_schedules=50)

Generated 50 random schedules
Makespan   range: [30.34, 53.18]
Variance   range: [0.09, 276.85]
Energy     range: [7.38, 11.81]

--- Reference: Low Makespan Target [30, 0.1, 9] ---
  Best schedule index: 0
  Best schedule -> makespan=33.68, variance=11.03, energy=8.66
  (z-scores: time=-1.09, var=-1.01, energy=-1.28)

--- Reference: Low Variance Target [40, 0.01, 9] ---
  Best schedule index: 0
  Best schedule -> makespan=33.68, variance=11.03, energy=8.66
  (z-scores: time=-1.09, var=-1.01, energy=-1.28)

--- Reference: Low Energy Target [45, 0.1, 7] ---
  Best schedule index: 24
  Best schedule -> makespan=40.25, variance=87.58, energy=7.38
  (z-scores: time=0.31, var=0.31, energy=-2.70)

=== Sanity Checks ===
Low-Makespan-Target picks lower/equal makespan than Low-Energy-Target: PASS
Low-Variance-Target picks below-median variance: PASS
Low-Energy-Target picks lower/equal energy than Low-Makespan-Target: PASS
Distinct 'best schedule' choices across 3 references: 2


## Test 4: Two-Headed Architecture

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# Load scenario
# ============================================================
def load_scenario(filepath="scenario.json"):
    with open(filepath, "r") as f:
        return json.load(f)

def zscore(x, axis=0, eps=1e-8):
    mean = x.mean(axis=axis, keepdims=True)
    std = x.std(axis=axis, keepdims=True) + eps
    return (x - mean) / std

def build_robot_features(scenario):
    pos = np.array(scenario["robot_pos"])
    battery = zscore(np.array(scenario["robot_battery"]).reshape(-1, 1))
    capacity = zscore(np.array(scenario["robot_max_payload"]).reshape(-1, 1))
    own_weight = zscore(np.array(scenario["robot_own_weight"]).reshape(-1, 1))
    return np.hstack([pos, battery, capacity, own_weight])

def build_task_features(scenario):
    pos = np.array(scenario["task_pos"])
    weight = zscore(np.array(scenario["task_weight"]).reshape(-1, 1))
    service_time = zscore(np.array(scenario["task_service_time"]).reshape(-1, 1))
    return np.hstack([pos, weight, service_time])


# ============================================================
# Sinkhorn assignment head (column-normalized only, from Test 2)
# ============================================================
def sinkhorn_column_normalize(logits, num_iters=20, temperature=0.5):
    log_alpha = logits / temperature
    for _ in range(num_iters):
        log_alpha = log_alpha - torch.logsumexp(log_alpha, dim=0, keepdim=True)  # column-normalize
    return torch.exp(log_alpha)


# ============================================================
# Two-headed model + NEW velocity head (3-headed):
#   1. Assignment head  -> which robot does which task (Sinkhorn, column-norm)
#   2. Rank/order head  -> continuous execution order value per cell
#   3. Velocity head    -> continuous velocity for each robot->task leg
#                          (i.e. velocity used when traveling TO that task)
# ============================================================
class ThreeHeadedDenoiser(nn.Module):
    def __init__(self, robot_feat_dim, task_feat_dim, d_model=64, nhead=4,
                 num_layers=2, pref_dim=3, min_vel=1.0, max_vel=10.0):
        super().__init__()
        self.robot_embed = nn.Linear(robot_feat_dim, d_model)
        self.task_embed = nn.Linear(task_feat_dim, d_model)
        self.pref_embed = nn.Linear(pref_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=d_model * 4,
            batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)

        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.cross_gate = nn.Parameter(torch.zeros(d_model))

        # Head 1: assignment logits (per robot-task cell, pre-Sinkhorn)
        self.assign_head = nn.Linear(d_model, 1)

        # Head 2: rank/order (continuous scalar per cell)
        self.rank_head = nn.Linear(d_model, 1)

        # Head 3: velocity via Beta distribution (predicts alpha, beta per cell)
        self.velocity_head = nn.Linear(d_model, 2)  # outputs raw alpha, beta params
        self.min_vel = min_vel
        self.max_vel = max_vel

    def forward(self, robot_feats, task_feats, pref):
        r_emb = self.robot_embed(robot_feats)   # (B,R,d)
        t_emb = self.task_embed(task_feats)      # (B,T,d)

        cell_ctx = r_emb.unsqueeze(2) + t_emb.unsqueeze(1)  # (B,R,T,d)
        B, R, T, D = cell_ctx.shape

        x = cell_ctx.view(B, R * T, D)
        x = self.encoder(x)

        pref_emb = self.pref_embed(pref).unsqueeze(1)
        cross_out, _ = self.cross_attn(x, pref_emb, pref_emb)
        x = x + self.cross_gate * cross_out

        x = x.view(B, R, T, D)

        assign_logits = self.assign_head(x).squeeze(-1)   # (B,R,T)
        rank_raw = self.rank_head(x).squeeze(-1)           # (B,R,T)
        
        vel_params = self.velocity_head(x)                      # (B,R,T,2)
        alpha = F.softplus(vel_params[..., 0]) + 1e-3             # ensure > 0
        beta = F.softplus(vel_params[..., 1]) + 1e-3               # ensure > 0

        beta_dist = torch.distributions.Beta(alpha, beta)
        vel_sample = beta_dist.rsample()                            # reparameterized sample, in (0,1)
        velocity = self.min_vel + (self.max_vel - self.min_vel) * vel_sample

        return assign_logits, rank_raw, velocity, alpha, beta


# ============================================================
# Decode: assignment (Sinkhorn -> hard argmax per task/column)
# + rank (only among cells assigned to that robot) -> valid order
# + velocity (read off directly for each assigned leg)
# ============================================================
def decode_schedule(assign_logits, rank_raw, velocity, temperature=0.5):
    B, R, T = assign_logits.shape
    assert B == 1
    assign_logits = assign_logits.squeeze(0)
    rank_raw = rank_raw.squeeze(0)
    velocity = velocity.squeeze(0)

    P = sinkhorn_column_normalize(assign_logits, temperature=temperature)  # (R,T)
    hard_assign = torch.argmax(P, dim=0)  # (T,) -> which robot each task goes to

    schedule = np.zeros((R, T), dtype=int)
    velocity_out = np.zeros((R, T), dtype=float)

    for r in range(R):
        task_ids = [t for t in range(T) if hard_assign[t].item() == r]
        if len(task_ids) == 0:
            continue
        # rank these tasks by their raw rank score -> valid contiguous order
        ranks = [rank_raw[r, t].item() for t in task_ids]
        order = np.argsort(ranks)  # ascending rank score defines execution order
        for pos, idx in enumerate(order):
            t = task_ids[idx]
            schedule[r, t] = pos + 1                      # 1..k, contiguous, no gaps
            velocity_out[r, t] = velocity[r, t].item()      # velocity for this leg

    return schedule, velocity_out, hard_assign.numpy()


# ============================================================
# TEST: Architectural shift validation
# ============================================================
def test_three_headed_architecture(json_path="scenario.json", seed=0):
    scenario = load_scenario(json_path)
    R, T = scenario["R"], scenario["T"]

    robot_feats_np = build_robot_features(scenario)
    task_feats_np = build_task_features(scenario)

    robot_feats = torch.tensor(robot_feats_np, dtype=torch.float32).unsqueeze(0)
    task_feats = torch.tensor(task_feats_np, dtype=torch.float32).unsqueeze(0)
    pref = torch.tensor([[0.34, 0.33, 0.33]], dtype=torch.float32)

    torch.manual_seed(seed)
    model = ThreeHeadedDenoiser(robot_feats_np.shape[1], task_feats_np.shape[1],
                                 min_vel=1.0, max_vel=10.0)
    model.eval()

    with torch.no_grad():
        assign_logits, rank_raw, velocity = model(robot_feats, task_feats, pref)

    schedule, velocity_out, hard_assign = decode_schedule(assign_logits, rank_raw, velocity)

    print(f"Generated schedule (R={R}, T={T}):\n{schedule}\n")
    print(f"Velocity per assigned leg (0 = unassigned cell):\n{np.round(velocity_out, 2)}\n")

    # --- Check 1: assignment validity (each task -> exactly one robot) ---
    tasks_per_robot_count = [np.sum(hard_assign == r) for r in range(R)]
    each_task_assigned_once = True  # guaranteed by construction (argmax over robots per task)
    print(f"Tasks per robot: {tasks_per_robot_count}")
    print(f"Each task assigned to exactly one robot (by construction): {each_task_assigned_once}")

    # --- Check 2: valid execution order (contiguous 1..k per robot, no duplicates/gaps) ---
    order_valid = True
    for r in range(R):
        orders = sorted(schedule[r][schedule[r] > 0])
        expected = list(range(1, len(orders) + 1))
        if orders != expected:
            order_valid = False
            print(f"  Robot {r} order INVALID: {orders} (expected {expected})")
    print(f"Execution order valid (contiguous, no gaps/dupes) for all robots: {order_valid}\n")

    # --- Check 3: velocity distinctiveness ---
    assigned_mask = schedule > 0
    assigned_velocities = velocity_out[assigned_mask]
    vel_std = assigned_velocities.std()
    vel_unique = len(np.unique(np.round(assigned_velocities, 3)))
    print(f"Assigned-leg velocities: {np.round(assigned_velocities, 3)}")
    print(f"Velocity std across legs: {vel_std:.4f}  (near 0 = velocities NOT distinct, a bug)")
    print(f"Number of distinct velocity values: {vel_unique} / {len(assigned_velocities)} legs")

    # --- Check 4: velocity bounds respected ---
    within_bounds = np.all((assigned_velocities >= model.min_vel - 1e-4) &
                            (assigned_velocities <= model.max_vel + 1e-4))
    print(f"All velocities within [{model.min_vel}, {model.max_vel}]: {within_bounds}")

    return schedule, velocity_out, order_valid, vel_std, within_bounds


if __name__ == "__main__":
    test_three_headed_architecture("scenarios/scenario_5_25_1001.json", seed=2)

Generated schedule (R=5, T=25):
[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0]
 [ 6  2 20  3 10 24 11  1 25  7 12  9 13 15  8  5  4 14 19 17 18 21 16 22
  23]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0]]

Velocity per assigned leg (0 = unassigned cell):
[[ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.  ]
 [ 9.95  5.86  9.64  9.32  4.94  3.7   9.69  2.62  9.99  7.96  9.97  9.47
   1.39  9.89  5.13  9.74  8.32  6.77  4.67 10.    9.03  5.5   6.56  3.88
  10.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0. 

## Test 5: Preference Conditioning

In [29]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# Loaders / features (same as before)
# ============================================================
def load_scenario(filepath="scenario.json"):
    with open(filepath, "r") as f:
        return json.load(f)

def zscore(x, axis=0, eps=1e-8):
    mean = x.mean(axis=axis, keepdims=True)
    std = x.std(axis=axis, keepdims=True) + eps
    return (x - mean) / std

def build_robot_features(scenario):
    pos = np.array(scenario["robot_pos"])
    battery = zscore(np.array(scenario["robot_battery"]).reshape(-1, 1))
    capacity = zscore(np.array(scenario["robot_max_payload"]).reshape(-1, 1))
    own_weight = zscore(np.array(scenario["robot_own_weight"]).reshape(-1, 1))
    return np.hstack([pos, battery, capacity, own_weight])

def build_task_features(scenario):
    pos = np.array(scenario["task_pos"])
    weight = zscore(np.array(scenario["task_weight"]).reshape(-1, 1))
    service_time = zscore(np.array(scenario["task_service_time"]).reshape(-1, 1))
    return np.hstack([pos, weight, service_time])


# ============================================================
# Sinkhorn (column-normalize only)
# ============================================================
def sinkhorn_column_normalize(logits, num_iters=20, temperature=0.5):
    log_alpha = logits / temperature
    for _ in range(num_iters):
        log_alpha = log_alpha - torch.logsumexp(log_alpha, dim=0, keepdim=True)
    return torch.exp(log_alpha)


# ============================================================
# Three-headed model (assignment + rank + Beta velocity)
# ============================================================
class ThreeHeadedDenoiser(nn.Module):
    def __init__(self, robot_feat_dim, task_feat_dim, d_model=64, nhead=4,
                 num_layers=2, pref_dim=3, min_vel=1.0, max_vel=10.0):
        super().__init__()
        self.robot_embed = nn.Linear(robot_feat_dim, d_model)
        self.task_embed = nn.Linear(task_feat_dim, d_model)
        self.pref_embed = nn.Linear(pref_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=d_model * 4,
            batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)

        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.cross_gate = nn.Parameter(torch.zeros(d_model))

        self.assign_head = nn.Linear(d_model, 1)
        self.rank_head = nn.Linear(d_model, 1)

        self.velocity_head = nn.Linear(d_model, 2)  # alpha, beta raw
        self.min_vel = min_vel
        self.max_vel = max_vel

    def forward(self, robot_feats, task_feats, pref):
        r_emb = self.robot_embed(robot_feats)
        t_emb = self.task_embed(task_feats)

        cell_ctx = r_emb.unsqueeze(2) + t_emb.unsqueeze(1)
        B, R, T, D = cell_ctx.shape

        x = cell_ctx.view(B, R * T, D)
        x = self.encoder(x)

        pref_emb = self.pref_embed(pref).unsqueeze(1)
        cross_out, _ = self.cross_attn(x, pref_emb, pref_emb)
        x = x + self.cross_gate * cross_out

        x = x.view(B, R, T, D)

        assign_logits = self.assign_head(x).squeeze(-1)
        rank_raw = self.rank_head(x).squeeze(-1)

        vel_params = self.velocity_head(x)
        alpha = F.softplus(vel_params[..., 0]) + 1e-3
        beta = F.softplus(vel_params[..., 1]) + 1e-3
        beta_dist = torch.distributions.Beta(alpha, beta)
        vel_sample = beta_dist.rsample()
        velocity = self.min_vel + (self.max_vel - self.min_vel) * vel_sample

        return assign_logits, rank_raw, velocity


def decode_schedule(assign_logits, rank_raw, velocity, temperature=0.5):
    B, R, T = assign_logits.shape
    assign_logits = assign_logits.squeeze(0)
    rank_raw = rank_raw.squeeze(0)
    velocity = velocity.squeeze(0)

    P = sinkhorn_column_normalize(assign_logits, temperature=temperature)
    hard_assign = torch.argmax(P, dim=0)

    schedule = np.zeros((R, T), dtype=int)
    velocity_out = np.zeros((R, T), dtype=float)

    for r in range(R):
        task_ids = [t for t in range(T) if hard_assign[t].item() == r]
        if len(task_ids) == 0:
            continue
        ranks = [rank_raw[r, t].item() for t in task_ids]
        order = np.argsort(ranks)
        for pos, idx in enumerate(order):
            t = task_ids[idx]
            schedule[r, t] = pos + 1
            velocity_out[r, t] = velocity[r, t].item()

    return schedule, velocity_out, hard_assign.numpy()


# ============================================================
# TEST 5: Preference Conditioning
# Fixed scenario + fixed random seed for the model's internal
# randomness (Beta sampling), vary ONLY the preference vector.
# ============================================================
def test_preference_conditioning(json_path="scenario.json", seed=0):
    scenario = load_scenario(json_path)
    R, T = scenario["R"], scenario["T"]

    robot_feats_np = build_robot_features(scenario)
    task_feats_np = build_task_features(scenario)
    robot_feats = torch.tensor(robot_feats_np, dtype=torch.float32).unsqueeze(0)
    task_feats = torch.tensor(task_feats_np, dtype=torch.float32).unsqueeze(0)

    torch.manual_seed(seed)
    model = ThreeHeadedDenoiser(robot_feats_np.shape[1], task_feats_np.shape[1])
    model.eval()

    with torch.no_grad():
        model.cross_gate.fill_(1.0)   # force gate open, diagnostic only — bypasses zero-init for this test

    preferences = {
        "Favor Time":     [0.8, 0.1, 0.1],
        "Favor Variance": [0.1, 0.8, 0.1],
        "Favor Energy":   [0.1, 0.1, 0.8],
        "Balanced":       [0.34, 0.33, 0.33],
    }

    schedules = {}
    velocities = {}
    assignments = {}

    for name, pref_vec in preferences.items():
        pref = torch.tensor([pref_vec], dtype=torch.float32)

        torch.manual_seed(seed)  # SAME seed each time -> isolates effect of pref only
        with torch.no_grad():
            assign_logits, rank_raw, velocity = model(robot_feats, task_feats, pref)

        schedule, velocity_out, hard_assign = decode_schedule(assign_logits, rank_raw, velocity)
        schedules[name] = schedule
        velocities[name] = velocity_out
        assignments[name] = hard_assign

        print(f"--- Preference: {name} {pref_vec} ---")
        print(f"Schedule:\n{schedule}")
        print(f"Task->Robot assignment: {hard_assign}")
        print(f"Mean velocity (assigned legs): {velocity_out[schedule > 0].mean():.3f}\n")

    # --- Sanity checks: do different preferences produce different outputs? ---
    print("=== Sanity Checks ===")

    names = list(preferences.keys())
    distinct_assignments = set()
    distinct_schedules = set()

    for name in names:
        distinct_assignments.add(tuple(assignments[name]))
        distinct_schedules.add(schedules[name].tobytes())

    print(f"Distinct task->robot assignment patterns across {len(names)} preferences: {len(distinct_assignments)}")
    print(f"Distinct full schedules across {len(names)} preferences: {len(distinct_schedules)}")

    # Compare velocity means pairwise
    print("\nMean velocity per preference:")
    for name in names:
        mean_vel = velocities[name][schedules[name] > 0].mean()
        print(f"  {name}: {mean_vel:.3f}")

    assignment_check = len(distinct_assignments) > 1
    schedule_check = len(distinct_schedules) > 1

    print(f"\nAssignment varies with preference: {'PASS' if assignment_check else 'FAIL (all identical)'}")
    print(f"Full schedule varies with preference: {'PASS' if schedule_check else 'FAIL (all identical)'}")

    return schedules, velocities, assignments


if __name__ == "__main__":
    test_preference_conditioning("scenarios\scenario_5_25_2002.json")

--- Preference: Favor Time [0.8, 0.1, 0.1] ---
Schedule:
[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0]
 [ 0 13  0  0  9  5 10 11  0  8 12  0  4  0  3  7  1  0  0  0  0  0  6  0
   2]
 [ 7  0 10  6  0  0  0  0 12  0  0  9  0 11  0  0  0  8  1  2  5  3  0  4
   0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0]]
Task->Robot assignment: [3 2 3 3 2 2 2 2 3 2 2 3 2 3 2 2 2 3 3 3 3 3 2 3 2]
Mean velocity (assigned legs): 5.773

--- Preference: Favor Variance [0.1, 0.8, 0.1] ---
Schedule:
[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0]
 [ 0 13  0  0  9  5 10 11  0  8 12  0  4  0  3  7  1  0  0  0  0  0  6  0
   2]
 [ 7  0 10  6  0  0  0  0 12  0  0  9  0 11  0  0  0  8  1  2  5  3  0  4
   0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 